# 🔮 ETS & ARIMA Forecasting
**fpppy Chapters 7–9 · Pattern Recognition for the Rest of Us**

> Two complementary forecasting families: ETS captures level, trend, and seasonality with exponential smoothing; ARIMA models autocorrelation structure after differencing. Together they cover most forecasting needs.

### Required reading
📘 [fpppy Ch 7 (ETS)](https://otexts.com/fpppy/expsmooth.html) · [Ch 8 (ARIMA)](https://otexts.com/fpppy/arima.html) · [Ch 9 (Evaluation)](https://otexts.com/fpppy/accuracy.html)

### What you'll learn
- Naive and seasonal naive baselines
- Simple Exponential Smoothing (SES), Holt's linear, Holt-Winters
- ARIMA: differencing, AR, MA — intuition and fitting
- auto_arima — automatic model selection
- Train/test split for time series, MAE/RMSE/MAPE evaluation

### Dataset: Air passengers + Retail sales
### Time: ~70 minutes

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install pmdarima -q
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Load data
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
passengers = pd.read_csv(url, header=0, names=['Month','Passengers'],
                          parse_dates=['Month'], index_col='Month')
passengers.index.freq = 'MS'  # monthly start frequency

# Train/test split — NEVER shuffle time series!
# Use last 24 months as test (2 years)
train = passengers.iloc[:-24]
test  = passengers.iloc[-24:]

print(f"Training: {len(train)} months ({train.index[0].strftime('%Y-%m')} to {train.index[-1].strftime('%Y-%m')})")
print(f"Test:     {len(test)} months ({test.index[0].strftime('%Y-%m')} to {test.index[-1].strftime('%Y-%m')})")
print("\n📌 CRITICAL: always split time series chronologically — never random!")

## 📐 Part 1 — Baseline Models

Before any sophisticated model, establish baselines. If your fancy model can't beat a naive baseline, something is wrong.

- **Naive:** forecast = last observed value
- **Seasonal Naive:** forecast = same period last year
- **Mean:** forecast = historical mean

These are your floor — any real model should do better.

In [ ]:
# Implement baseline forecasts
h = len(test)

naive_fc      = pd.Series([train['Passengers'].iloc[-1]] * h, index=test.index)
seasonal_naive = train['Passengers'].iloc[-12:].values  # last 12 months repeated
snaive_fc     = pd.Series(np.tile(seasonal_naive, 2)[:h], index=test.index)
mean_fc       = pd.Series([train['Passengers'].mean()] * h, index=test.index)

def eval_forecast(name, fc, actual):
    mae  = mean_absolute_error(actual, fc)
    rmse = np.sqrt(mean_squared_error(actual, fc))
    mape = np.mean(np.abs((actual - fc) / actual)) * 100
    return {'Model':name, 'MAE':mae, 'RMSE':rmse, 'MAPE%':mape}

results = []
for name, fc in [('Naive', naive_fc), ('Seasonal Naive', snaive_fc), ('Mean', mean_fc)]:
    results.append(eval_forecast(name, fc, test['Passengers']))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual', alpha=0.8)
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
for fc, color, name in [(naive_fc,'#e85d20','Naive'),
                         (snaive_fc,'#1e5fa8','Seasonal Naive'),
                         (mean_fc,'#888','Mean')]:
    ax.plot(fc.index, fc, color=color, lw=1.5, ls='--', label=name)
ax.set_title('Baseline Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

print("\n=== Baseline Performance ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\n📌 Seasonal Naive is surprisingly hard to beat on strongly seasonal data")

## 📈 Part 2 — Exponential Smoothing (ETS)

ETS models assign exponentially decreasing weights to past observations — recent observations matter more.

- **SES (α):** level only — best for flat series
- **Holt (α, β):** level + trend — best for trending series  
- **Holt-Winters (α, β, γ):** level + trend + seasonality — the full model

The 'ETS' name refers to Error-Trend-Seasonality. Each component can be None, Additive, or Multiplicative.

In [ ]:
# Fit Holt-Winters (additive and multiplicative)
hw_add  = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='add', seasonal_periods=12).fit()
hw_mult = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='mul', seasonal_periods=12).fit()

fc_add  = hw_add.forecast(h)
fc_mult = hw_mult.forecast(h)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--')
ax.plot(fc_add.index,  fc_add,  color='#1e5fa8', lw=2, label='Holt-Winters Additive')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=2, label='Holt-Winters Multiplicative')
ax.fill_between(test.index,
                fc_mult * 0.92, fc_mult * 1.08,
                alpha=0.15, color='#e85d20', label='~80% interval (multiplicative)')
ax.set_title('Holt-Winters Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

results += [eval_forecast('HW Additive',       fc_add,  test['Passengers']),
            eval_forecast('HW Multiplicative',  fc_mult, test['Passengers'])]

print("\n=== ETS vs Baseline ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\nSmoothing parameters (multiplicative):")
print(f"  α (level) = {hw_mult.params['smoothing_level']:.3f}")
print(f"  β (trend) = {hw_mult.params['smoothing_trend']:.3f}")
print(f"  γ (seasonal) = {hw_mult.params['smoothing_seasonal']:.3f}")

## 📉 Part 3 — ARIMA: AutoRegressive Integrated Moving Average

**AR(p):** y_t depends on its past p values
```
y_t = φ₁y_{t-1} + φ₂y_{t-2} + ... + φₚy_{t-p} + ε_t
```

**MA(q):** y_t depends on past q forecast errors
```
y_t = ε_t + θ₁ε_{t-1} + ... + θqε_{t-q}
```

**I(d):** differencing d times to achieve stationarity

**ARIMA(p,d,q)** combines all three. **SARIMA(p,d,q)(P,D,Q)m** adds seasonal components.

In [ ]:
# auto_arima — finds optimal (p,d,q)(P,D,Q) automatically
print("Running auto_arima (may take 30-60 seconds)...")
arima_model = auto_arima(
    train['Passengers'],
    seasonal=True,
    m=12,           # monthly seasonality
    stepwise=True,  # faster search
    information_criterion='aic',
    trace=False,
    error_action='ignore',
    suppress_warnings=True
)
print("\nBest ARIMA model found:")
print(arima_model.summary())

In [ ]:
# Forecast with best ARIMA
arima_fc_vals, conf_int = arima_model.predict(n_periods=h, return_conf_int=True)
arima_fc = pd.Series(arima_fc_vals, index=test.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
ax.plot(arima_fc.index, arima_fc, color='#1a7a45', lw=2, label=f'ARIMA{arima_model.order}{arima_model.seasonal_order}')
ax.fill_between(test.index, conf_int[:,0], conf_int[:,1], alpha=0.15, color='#1a7a45', label='95% CI')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=1.5, ls='--', alpha=0.7, label='HW Multiplicative')
ax.set_title('ARIMA vs Holt-Winters — Air Passengers Forecast')
ax.legend()
plt.tight_layout()
plt.show()

results.append(eval_forecast('ARIMA', arima_fc, test['Passengers']))
print("\n=== Final Model Comparison ===")
df_res = pd.DataFrame(results).sort_values('RMSE')
print(df_res.to_string(index=False, float_format='%.2f'))
print(f"\n🏆 Best model by RMSE: {df_res.iloc[0]['Model']}")

In [ ]:
# Residual diagnostics — good model → white noise residuals
arima_model.plot_diagnostics(figsize=(12, 8))
plt.suptitle('ARIMA Residual Diagnostics\n(residuals should look like white noise)', y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 What to look for:")
print("  Standardized residuals: random scatter around zero, no patterns")
print("  Histogram: approximately normal")
print("  Q-Q plot: points on the diagonal")
print("  Correlogram: no significant spikes (residuals uncorrelated)")

In [ ]:
answers={"q1":"","q2":"","q3":"","q4":"","q5":""}
# q1: What do the three parameters of ARIMA(p,d,q) each represent?
# q2: Why can you NOT randomly shuffle train/test split for time series?
# q3: When would you choose Holt-Winters over ARIMA?
# q4: What does the 'd' in ARIMA represent and when is d=1 used?
# q5: If residuals show significant ACF spikes, what does that indicate?
missing=[k for k,v in answers.items() if not v.strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*